# Generate InstructPix2Pix Fine-Tuning Image Pairs

Generate paired editing data for later InstructPix2Pix fine-tuning. The notebook uses deterministic source-image selection from the same `traditional_houses` dataset used by the grid notebooks, then edits each source image toward the `gadang` concept with two inpainting models:

1. `flux_fill_nf4`
2. `sdxl_inpainting`

Each record contains `input_image`, `mask_image`, `edited_image`, and `edit_prompt`.


In [ ]:
from __future__ import annotations

import os
import sys
import re
import json
import random
import traceback
from pathlib import Path
from datetime import datetime


def env_int(name: str, default: int) -> int:
    value = os.environ.get(name, "").strip()
    return int(value) if value else default


def env_float(name: str, default: float) -> float:
    value = os.environ.get(name, "").strip()
    return float(value) if value else default


def env_str(name: str, default: str) -> str:
    value = os.environ.get(name, "").strip()
    return value if value else default


def slugify(value: str) -> str:
    value = value.replace(".", "p").replace("-", "m")
    return re.sub(r"[^A-Za-z0-9_]+", "_", value).strip("_")


PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

REQUESTED_DATA_ROOT = PROJECT_ROOT / "data" / "traditional_houses"
FALLBACK_DATA_ROOT = PROJECT_ROOT / "traditional_houses"
DATA_ROOT = REQUESTED_DATA_ROOT if REQUESTED_DATA_ROOT.exists() else FALLBACK_DATA_ROOT

CONCEPT_TOKEN = env_str("GADANG_CONCEPT_TOKEN", "gadang")
TARGET_CLASS = "Gadang"
HOUSE_CLASSES = ["Gadang", "Joglo", "Honai", "Panjang", "Tongkonan"]
CLASS_ALIASES = {name.lower(): name for name in HOUSE_CLASSES}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

SOURCE_IMAGE_COUNT = env_int("GADANG_GENERATE_SOURCE_IMAGES", 50)
GENERATION_RESOLUTION = env_int("GADANG_GENERATION_RESOLUTION", 1024)
SEED = env_int("GADANG_SEED", 100)
GENERATION_MODELS = env_str("GADANG_GENERATION_MODELS", "flux_fill_nf4 sdxl_inpainting").split()
NUM_INFERENCE_STEPS = env_int("GADANG_GENERATION_STEPS", 30)
GUIDANCE_SCALE = env_float("GADANG_GENERATION_GUIDANCE_SCALE", 7.5)
STRENGTH = env_float("GADANG_GENERATION_STRENGTH", 0.95)
MASK_LEFT = env_float("GADANG_MASK_LEFT", 0.10)
MASK_TOP = env_float("GADANG_MASK_TOP", 0.16)
MASK_RIGHT = env_float("GADANG_MASK_RIGHT", 0.90)
MASK_BOTTOM = env_float("GADANG_MASK_BOTTOM", 0.94)

FLUX_FILL_DIR = Path(env_str("GADANG_FLUX_FILL_DIR", str(PROJECT_ROOT / "models" / "flux-fill-nf4")))
FLUX_BASE_MODEL = env_str("GADANG_FLUX_BASE_MODEL", "black-forest-labs/FLUX.1-dev")
SDXL_INPAINT_CKPT = Path(env_str("GADANG_SDXL_INPAINT_CKPT", str(PROJECT_ROOT / "models" / "sdxl-inpainting" / "sd_xl_base_1.0_inpainting_0.1.safetensors")))

RUN_STAMP = env_str("GADANG_RUN_STAMP", datetime.now().strftime("%Y%m%d_%H%M%S"))
DEFAULT_RUN_NAME = f"{RUN_STAMP}_train_pix2pix_dataset_img{SOURCE_IMAGE_COUNT}"
RUN_NAME = slugify(env_str("GADANG_GENERATOR_RUN_NAME", DEFAULT_RUN_NAME))

OUTPUT_DIR = PROJECT_ROOT / "outputs" / RUN_NAME
DATASET_DIR = OUTPUT_DIR / "generated_instructpix2pix_dataset"
INPUT_DIR = DATASET_DIR / "input_images"
MASK_DIR = DATASET_DIR / "masks"
EDITED_DIR = DATASET_DIR / "edited_images"
COMPARISON_DIR = DATASET_DIR / "comparison_grids"
LOG_DIR = OUTPUT_DIR / "logs"
NOTEBOOK_OUTPUT_DIR = OUTPUT_DIR / "notebook"
METADATA_PATH = DATASET_DIR / "metadata.jsonl"
MANIFEST_PATH = DATASET_DIR / "generation_manifest.json"
RUN_CONFIG_PATH = OUTPUT_DIR / "run_config.json"
ARTIFACT_LOG_PATH = OUTPUT_DIR / "artifact_log.json"

for directory in [OUTPUT_DIR, DATASET_DIR, INPUT_DIR, MASK_DIR, EDITED_DIR, COMPARISON_DIR, LOG_DIR, NOTEBOOK_OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

ROOT_LOG_PATH = PROJECT_ROOT / "output.log"
RUN_LOG_PATH = LOG_DIR / "run_output.log"
NOTEBOOK_WRITES_ROOT_LOG = env_int("GADANG_NOTEBOOK_WRITES_ROOT_LOG", 0) == 1

class Tee:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, data):
        for stream in self.streams:
            stream.write(data)
            stream.flush()

    def flush(self):
        for stream in self.streams:
            stream.flush()

_run_log_file = RUN_LOG_PATH.open("w", encoding="utf-8")
if NOTEBOOK_WRITES_ROOT_LOG:
    _root_log_file = ROOT_LOG_PATH.open("w", encoding="utf-8")
    sys.stdout = Tee(sys.__stdout__, _root_log_file, _run_log_file)
    sys.stderr = Tee(sys.__stderr__, _root_log_file, _run_log_file)
else:
    sys.stdout = Tee(sys.__stdout__, _run_log_file)
    sys.stderr = Tee(sys.__stderr__, _run_log_file)

random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

RUN_CONFIG = {
    "method": "generate_train_pix2pix_pairs",
    "run_name": RUN_NAME,
    "run_stamp": RUN_STAMP,
    "seed": SEED,
    "source_image_count": SOURCE_IMAGE_COUNT,
    "generation_resolution": GENERATION_RESOLUTION,
    "generation_models": GENERATION_MODELS,
    "num_inference_steps": NUM_INFERENCE_STEPS,
    "guidance_scale": GUIDANCE_SCALE,
    "strength": STRENGTH,
    "dataset_root": str(DATA_ROOT),
    "flux_fill_dir": str(FLUX_FILL_DIR),
    "flux_base_model": FLUX_BASE_MODEL,
    "sdxl_inpaint_checkpoint": str(SDXL_INPAINT_CKPT),
    "metadata_path": str(METADATA_PATH),
}
RUN_CONFIG_PATH.write_text(json.dumps(RUN_CONFIG, indent=2), encoding="utf-8")


def log_event(message: str) -> None:
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {message}", flush=True)


def log_section(title: str) -> None:
    log_event("=" * 72)
    log_event(title)
    log_event("=" * 72)

log_section("Generator notebook initialized")
log_event(f"Project root: {PROJECT_ROOT}")
log_event(f"Dataset root: {DATA_ROOT}")
log_event(f"Output dir: {OUTPUT_DIR}")
print(json.dumps(RUN_CONFIG, indent=2), flush=True)
assert DATA_ROOT.exists(), f"Dataset directory was not found: {DATA_ROOT}"


In [ ]:
from PIL import Image, ImageOps, ImageDraw, ImageFilter
import pandas as pd
import numpy as np
from tqdm.auto import tqdm


def canonical_class_name(path: Path) -> str:
    key = path.name.lower()
    return CLASS_ALIASES.get(key, path.name.capitalize())


def discover_images(data_root: Path):
    rows = []
    for class_dir in sorted([p for p in data_root.iterdir() if p.is_dir()]):
        class_name = canonical_class_name(class_dir)
        if class_name not in HOUSE_CLASSES:
            continue
        for image_path in sorted(class_dir.rglob("*")):
            if image_path.suffix.lower() in IMAGE_EXTENSIONS:
                rows.append({"class_name": class_name, "image_path": str(image_path)})
    return pd.DataFrame(rows)


def resize_square(image: Image.Image, size: int = GENERATION_RESOLUTION) -> Image.Image:
    image = ImageOps.exif_transpose(image).convert("RGB")
    width, height = image.size
    crop_size = min(width, height)
    left = (width - crop_size) // 2
    top = (height - crop_size) // 2
    image = image.crop((left, top, left + crop_size, top + crop_size))
    return image.resize((size, size), Image.Resampling.LANCZOS)


def create_house_edit_mask(size: int = GENERATION_RESOLUTION) -> Image.Image:
    mask = Image.new("L", (size, size), 0)
    draw = ImageDraw.Draw(mask)
    box = (
        int(size * MASK_LEFT),
        int(size * MASK_TOP),
        int(size * MASK_RIGHT),
        int(size * MASK_BOTTOM),
    )
    draw.rounded_rectangle(box, radius=max(size // 24, 12), fill=255)
    return mask.filter(ImageFilter.GaussianBlur(radius=max(size // 80, 6)))


images_df = discover_images(DATA_ROOT)
print(images_df.groupby("class_name").size())
assert len(images_df) > 0, "No source images found."

source_df = images_df[images_df["class_name"] != TARGET_CLASS].sort_values(["class_name", "image_path"]).reset_index(drop=True)
assert len(source_df) > 0, "No non-Gadang source images found for editing."

# Balanced deterministic sampling across non-Gadang classes.
selected_rows = []
class_names = [name for name in HOUSE_CLASSES if name != TARGET_CLASS]
class_groups = {
    class_name: source_df[source_df["class_name"] == class_name].sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    for class_name in class_names
}
cursor = {class_name: 0 for class_name in class_names}
while len(selected_rows) < SOURCE_IMAGE_COUNT:
    progressed = False
    for class_name in class_names:
        group = class_groups[class_name]
        if len(group) == 0:
            continue
        idx = cursor[class_name] % len(group)
        selected_rows.append(group.iloc[idx].to_dict())
        cursor[class_name] += 1
        progressed = True
        if len(selected_rows) >= SOURCE_IMAGE_COUNT:
            break
    if not progressed:
        break

selected_df = pd.DataFrame(selected_rows).reset_index(drop=True)
assert len(selected_df) == SOURCE_IMAGE_COUNT, f"Expected {SOURCE_IMAGE_COUNT} selected source images, got {len(selected_df)}"

EDIT_PROMPTS = [
    "replace the main house with a realistic gadang traditional house from West Sumatra, iconic curved gonjong roof, natural lighting",
    "transform the building into a Rumah Gadang Minangkabau house with dramatic curved roof horns, photorealistic architecture",
    "edit this scene so the central building becomes gadang traditional architecture, wooden facade, curved gonjong roof",
    "make the main house look like an authentic gadang house from West Sumatra while preserving the surrounding scene",
]

source_manifest = []
for idx, row in selected_df.iterrows():
    source_path = Path(row["image_path"])
    image_id = f"source_{idx:05d}_{row['class_name'].lower()}"
    input_path = INPUT_DIR / f"{image_id}_input.png"
    mask_path = MASK_DIR / f"{image_id}_mask.png"
    input_image = resize_square(Image.open(source_path), GENERATION_RESOLUTION)
    mask_image = create_house_edit_mask(GENERATION_RESOLUTION)
    input_image.save(input_path)
    mask_image.save(mask_path)
    source_manifest.append({
        "image_id": image_id,
        "class_name": row["class_name"],
        "source_image": str(source_path),
        "input_image": str(input_path),
        "mask_image": str(mask_path),
        "edit_prompt": EDIT_PROMPTS[idx % len(EDIT_PROMPTS)],
    })

source_manifest_path = DATASET_DIR / "source_manifest.json"
source_manifest_path.write_text(json.dumps(source_manifest, indent=2), encoding="utf-8")
print(f"Prepared {len(source_manifest)} source images and masks.")
print(f"Saved source manifest: {source_manifest_path}")


In [ ]:
import torch
from diffusers import StableDiffusionXLInpaintPipeline


def get_dtype(prefer_bfloat16: bool = False):
    if not torch.cuda.is_available():
        return torch.float32
    if prefer_bfloat16 and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float16


def maybe_enable_memory_savers(pipe):
    if torch.cuda.is_available():
        try:
            pipe.enable_model_cpu_offload()
        except Exception:
            pipe = pipe.to("cuda")
        try:
            pipe.enable_attention_slicing()
        except Exception:
            pass
    return pipe


def load_sdxl_inpainting_pipeline():
    assert SDXL_INPAINT_CKPT.exists(), f"SDXL inpainting checkpoint not found: {SDXL_INPAINT_CKPT}"
    dtype = get_dtype(prefer_bfloat16=False)
    log_event(f"Loading local SDXL inpainting checkpoint: {SDXL_INPAINT_CKPT}")
    pipe = StableDiffusionXLInpaintPipeline.from_single_file(
        str(SDXL_INPAINT_CKPT),
        torch_dtype=dtype,
        safety_checker=None,
    )
    return maybe_enable_memory_savers(pipe)


def load_flux_fill_nf4_pipeline():
    from diffusers import DiffusionPipeline, FluxFillPipeline, FluxTransformer2DModel
    from transformers import T5EncoderModel

    assert FLUX_FILL_DIR.exists(), f"Flux fill directory not found: {FLUX_FILL_DIR}"
    dtype = get_dtype(prefer_bfloat16=True)
    log_event(f"Loading local Flux NF4 components from: {FLUX_FILL_DIR}")
    log_event(f"Loading Flux base pipeline from local cache only: {FLUX_BASE_MODEL}")
    orig_pipeline = DiffusionPipeline.from_pretrained(
        FLUX_BASE_MODEL,
        torch_dtype=dtype,
        local_files_only=True,
    )
    transformer = FluxTransformer2DModel.from_pretrained(
        str(FLUX_FILL_DIR),
        subfolder="transformer",
        torch_dtype=dtype,
        local_files_only=True,
    )
    text_encoder_2 = T5EncoderModel.from_pretrained(
        str(FLUX_FILL_DIR),
        subfolder="text_encoder_2",
        torch_dtype=dtype,
        local_files_only=True,
    )
    pipe = FluxFillPipeline.from_pipe(
        orig_pipeline,
        transformer=transformer,
        text_encoder_2=text_encoder_2,
        torch_dtype=dtype,
    )
    return maybe_enable_memory_savers(pipe)


PIPELINE_LOADERS = {
    "sdxl_inpainting": load_sdxl_inpainting_pipeline,
    "flux_fill_nf4": load_flux_fill_nf4_pipeline,
}


In [ ]:
def generate_with_sdxl(pipe, prompt: str, image: Image.Image, mask: Image.Image, seed: int):
    generator_device = "cuda" if torch.cuda.is_available() else "cpu"
    generator = torch.Generator(device=generator_device).manual_seed(seed)
    return pipe(
        prompt=prompt,
        image=image,
        mask_image=mask,
        strength=STRENGTH,
        num_inference_steps=NUM_INFERENCE_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        generator=generator,
    ).images[0]


def generate_with_flux(pipe, prompt: str, image: Image.Image, mask: Image.Image, seed: int):
    generator_device = "cuda" if torch.cuda.is_available() else "cpu"
    generator = torch.Generator(device=generator_device).manual_seed(seed)
    return pipe(
        prompt=prompt,
        image=image,
        mask_image=mask,
        height=GENERATION_RESOLUTION,
        width=GENERATION_RESOLUTION,
        num_inference_steps=NUM_INFERENCE_STEPS,
        max_sequence_length=512,
        generator=generator,
    ).images[0]


GENERATION_FNS = {
    "sdxl_inpainting": generate_with_sdxl,
    "flux_fill_nf4": generate_with_flux,
}


def save_comparison_grid(record: dict):
    from PIL import ImageDraw

    input_image = Image.open(record["input_image"]).convert("RGB")
    mask_image = Image.open(record["mask_image"]).convert("RGB")
    edited_image = Image.open(record["edited_image"]).convert("RGB")
    w, h = input_image.size
    header = 52
    grid = Image.new("RGB", (w * 3, h + header), "white")
    draw = ImageDraw.Draw(grid)
    titles = ["input", "mask", record["generator_model"]]
    for col, (title, image) in enumerate(zip(titles, [input_image, mask_image, edited_image])):
        x = col * w
        grid.paste(image, (x, header))
        draw.text((x + 12, 16), title, fill=(0, 0, 0))
    out_path = COMPARISON_DIR / record["generator_model"] / f"{record['image_id']}_{record['generator_model']}_comparison.png"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    grid.save(out_path)
    return out_path


records = []
model_status = {}
for model_name in GENERATION_MODELS:
    if model_name not in PIPELINE_LOADERS:
        model_status[model_name] = {"status": "skipped", "reason": "unknown model key"}
        log_event(f"Skipping unknown generation model: {model_name}")
        continue

    log_section(f"Generation model: {model_name}")
    try:
        pipe = PIPELINE_LOADERS[model_name]()
        model_status[model_name] = {"status": "loaded"}
    except Exception as exc:
        model_status[model_name] = {
            "status": "failed_to_load",
            "error": repr(exc),
            "traceback": traceback.format_exc(),
        }
        log_event(f"Failed to load {model_name}: {exc}")
        print(traceback.format_exc(), flush=True)
        continue

    model_dir = EDITED_DIR / model_name
    model_dir.mkdir(parents=True, exist_ok=True)
    for idx, source in enumerate(tqdm(source_manifest, desc=f"Generating {model_name}")):
        image = Image.open(source["input_image"]).convert("RGB")
        mask = Image.open(source["mask_image"]).convert("L")
        prompt = source["edit_prompt"]
        seed = SEED + idx
        edited_path = model_dir / f"{source['image_id']}_{model_name}_edited.png"
        try:
            edited = GENERATION_FNS[model_name](pipe, prompt, image, mask, seed)
            edited.save(edited_path)
            record = {
                "run_name": RUN_NAME,
                "image_id": source["image_id"],
                "generator_model": model_name,
                "class_name": source["class_name"],
                "source_image": source["source_image"],
                "input_image": source["input_image"],
                "mask_image": source["mask_image"],
                "edited_image": str(edited_path),
                "edit_prompt": prompt,
                "seed": seed,
                "resolution": GENERATION_RESOLUTION,
                "status": "ok",
            }
            record["comparison_grid"] = str(save_comparison_grid(record))
            records.append(record)
        except Exception as exc:
            record = {
                "run_name": RUN_NAME,
                "image_id": source["image_id"],
                "generator_model": model_name,
                "class_name": source["class_name"],
                "source_image": source["source_image"],
                "input_image": source["input_image"],
                "mask_image": source["mask_image"],
                "edited_image": None,
                "edit_prompt": prompt,
                "seed": seed,
                "resolution": GENERATION_RESOLUTION,
                "status": "failed_generation",
                "error": repr(exc),
            }
            records.append(record)
            log_event(f"Failed generation {model_name} {source['image_id']}: {exc}")
            print(traceback.format_exc(), flush=True)

    del pipe
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

with METADATA_PATH.open("w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record) + "\n")

manifest = {
    "run_config": RUN_CONFIG,
    "model_status": model_status,
    "source_manifest": source_manifest,
    "record_count": len(records),
    "successful_record_count": sum(1 for r in records if r.get("status") == "ok"),
    "metadata_path": str(METADATA_PATH),
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Saved metadata: {METADATA_PATH}")
print(f"Saved generation manifest: {MANIFEST_PATH}")
print(json.dumps(model_status, indent=2), flush=True)


In [ ]:
artifact_log = {
    "run_name": RUN_NAME,
    "output_dir": str(OUTPUT_DIR),
    "dataset_dir": str(DATASET_DIR),
    "metadata_path": str(METADATA_PATH),
    "manifest_path": str(MANIFEST_PATH),
    "run_config_path": str(RUN_CONFIG_PATH),
    "run_output_log": str(RUN_LOG_PATH),
    "comparison_dir": str(COMPARISON_DIR),
    "edited_dir": str(EDITED_DIR),
}
ARTIFACT_LOG_PATH.write_text(json.dumps(artifact_log, indent=2), encoding="utf-8")
print(json.dumps(artifact_log, indent=2), flush=True)
log_section("Generator notebook finished")
